# Los datos

De dónde sale la base, qué se limpia y con qué registros se trabaja en el resto del libro.
Al final de este capítulo quedan 15.459 viviendas, y todos los capítulos siguientes
parten de esa misma base.

In [1]:
import io, math, urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from PIL import Image
from scipy import stats
from sklearn.neighbors import NearestNeighbors, KNeighborsClassifier
from sklearn.model_selection import GroupKFold, cross_val_score

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

AZUL, ROJO, VERDE, GRIS = "#2F6F9F", "#C7522A", "#6A9B4F", "#9FB8CC"
ORDEN_ESTRATO = ["Bajo_Bajo_1", "Bajo_2", "Medio_Bajo_3", "Medio_4", "Medio_Alto_5", "Alto_6", "No_Aplica", "Otro"]
ESTRATOS = ORDEN_ESTRATO[:6]
NOMBRES_ESTRATO = [f"Estrato {i}" for i in range(1, 7)]
COLORES_ESTRATO = ["#253494", "#2C7FB8", "#41B6C4", "#FDAE61", "#F46D43", "#A50026"]
NO_BARRIOS = ["RURAL", "ZONA NO DESARROLLADA", "SIN LOCALIDAD"]

# Gráficos interactivos: el formato que lee VS Code y HTML para Jupyter
pio.renderers.default = "plotly_mimetype+notebook_connected"
pio.templates["eda"] = go.layout.Template(layout=dict(
    font=dict(family="Segoe UI, Arial", size=13, color="#222"),
    colorway=[AZUL, ROJO, VERDE, "#E0A030", "#7B5EA7", "#3BA3A3"],
    title=dict(x=0.01, font=dict(size=17)),
    margin=dict(l=60, r=30, t=80, b=50),
    xaxis=dict(gridcolor="#EEEEEE", zeroline=False),
    yaxis=dict(gridcolor="#EEEEEE", zeroline=False),
    hoverlabel=dict(font_family="Segoe UI, Arial"),
))
pio.templates.default = "plotly_white+eda"

def cop(x):
    """Plata en formato legible."""
    if pd.isna(x):
        return "-"
    if abs(x) >= 1e9:
        return f"${x/1e9:,.2f} mil M"
    if abs(x) >= 1e6:
        return f"${x/1e6:,.1f} M"
    if abs(x) >= 1e3:
        return f"${x/1e3:,.0f} mil"
    return f"${x:,.0f}"

def mostrar(fig, titulo, alto=450):
    fig.update_layout(title=titulo, height=alto)
    fig.show()

## Diccionario de variables

Estas son las variables que usa el análisis. Las del final las calculamos en este capítulo.

In [2]:
variables = [
    # grupo, variable, tipo, valores o unidad, descripción
    ("Identificación", "fuente", "Categórica nominal", "observatorio, georreferenciadas, igac, avisos_registro", "Base original de la compraventa."),
    ("", "anio", "Numérica discreta", "2016 – 2025", "Año de la compraventa."),
    ("Precio y tamaño", "precio", "Numérica continua", "pesos (COP)", "Valor pagado. Es la variable que queremos explicar."),
    ("", "area", "Numérica continua", "m²", "Área construida."),
    ("", "precio_m2", "Numérica continua", "COP por m²", "Precio dividido por el área."),
    ("", "avaluo", "Numérica continua", "pesos (COP)", "Avalúo catastral del predio."),
    ("Ubicación", "lat, lon", "Numérica continua", "grados", "Coordenadas del predio."),
    ("", "barrio", "Categórica nominal", "176 barrios", "Barrio donde queda el predio."),
    ("", "localidad", "Categórica nominal", "6 localidades", "Localidad del distrito."),
    ("", "estrato", "Categórica ordinal", "1 a 6, No_Aplica, Otro", "Estrato socioeconómico."),
    ("Características", "habitaciones, banios", "Numérica discreta", "unidades", "Número de habitaciones y de baños."),
    ("", "piso", "Numérica discreta", "1, 2, 3…", "Piso de la unidad."),
    ("", "anio_construccion", "Numérica discreta", "año", "Año de construcción."),
    ("", "condicion_predio", "Categórica nominal", "NPH, PH…", "NPH = casa; PH = unidad en propiedad horizontal."),
    ("", "clase_unidad_principal", "Categórica nominal", "vivienda, anexo, otro_uso", "Uso de la unidad. Solo analizamos vivienda."),
    ("", "origen_caract", "Categórica nominal", "unidad, edificio", "Si el área se midió en la unidad o se tomó del edificio."),
    ("Calidad", "apto_modelo", "Binaria", "True / False", "Pasa los filtros de calidad de la base."),
    ("", "precio_m2_en_rango", "Binaria", "True / False", "Precio por m² entre 300 mil y 25 M."),
    ("Calculadas aquí", "precio_real, precio_m2_real", "Numérica continua", "pesos de dic. 2025", "Precios ajustados por inflación."),
    ("", "edad", "Numérica discreta", "años", "Años entre la construcción y la venta."),
    ("", "razon", "Numérica continua", "veces", "Precio (en pesos de 2022) dividido por el avalúo."),
]
diccionario = pd.DataFrame(variables, columns=["Grupo", "Variable", "Tipo", "Valores o unidad", "Descripción"])
inicio_grupo = diccionario.Grupo.ne("")

def linea_de_grupo(fila):
    borde = "2px solid #1F3B57" if inicio_grupo[fila.name] else "1px solid #E3E8EE"
    return [f"border-top: {borde}"] * len(fila)

(diccionario.style
    .hide(axis="index")
    .apply(linea_de_grupo, axis=1)
    .set_properties(**{"text-align": "left", "vertical-align": "top", "padding": "5px 10px",
                       "font-size": "12.5px", "background-color": "white", "color": "#222"})
    .set_properties(subset=["Grupo"], **{"font-weight": "bold", "color": "#1F3B57"})
    .set_properties(subset=["Variable"], **{"font-family": "Consolas, monospace", "color": "#9C3D1A"})
    .set_table_styles([
        {"selector": "", "props": "border-collapse: collapse; font-family: 'Segoe UI', Arial, sans-serif;"},
        {"selector": "th", "props": "background-color: #1F3B57; color: white; text-align: left; padding: 7px 10px;"},
    ]))

Grupo,Variable,Tipo,Valores o unidad,Descripción
Identificación,fuente,Categórica nominal,"observatorio, georreferenciadas, igac, avisos_registro",Base original de la compraventa.
,anio,Numérica discreta,2016 – 2025,Año de la compraventa.
Precio y tamaño,precio,Numérica continua,pesos (COP),Valor pagado. Es la variable que queremos explicar.
,area,Numérica continua,m²,Área construida.
,precio_m2,Numérica continua,COP por m²,Precio dividido por el área.
,avaluo,Numérica continua,pesos (COP),Avalúo catastral del predio.
Ubicación,"lat, lon",Numérica continua,grados,Coordenadas del predio.
,barrio,Categórica nominal,176 barrios,Barrio donde queda el predio.
,localidad,Categórica nominal,6 localidades,Localidad del distrito.
,estrato,Categórica ordinal,"1 a 6, No_Aplica, Otro",Estrato socioeconómico.


## Carga y limpieza

In [3]:
df = pd.read_csv("datos/base_final_excel_es.csv", sep=";", decimal=",", low_memory=False)
n_total = len(df)

for col in ["manzana", "codigo_predial"]:
    df[col] = (df[col].astype("string").str.replace('="', "", regex=False)
                      .str.replace('"', "", regex=False).str.strip())
# estas dos banderas traen vacíos y pandas las lee como texto
for col in ["area_es_suma_pisos", "area_confiable"]:
    df[col] = df[col].map({True: True, False: False, "True": True, "False": False}).astype("boolean")
for col in [c for c in df.columns if str(df[c].dtype) in ("object", "str", "string")]:
    df[col] = df[col].astype("string").str.strip().replace({"": pd.NA})

cobertura = (df.assign(con_coordenadas=df.lat.notna(), con_area=df.area.notna(), con_estrato=df.estrato.notna())
               .groupby("fuente")[["con_coordenadas", "con_area", "con_estrato"]].mean().mul(100).round(1))
cobertura.insert(0, "registros", df.fuente.value_counts())
cobertura

,registros,con_coordenadas,con_area,con_estrato
fuente,,,,
avisos_registro,427,0.00,98.60,0.00
georreferenciadas,5351,95.90,93.70,92.80
igac,40547,12.50,12.30,12.50
observatorio,22384,100.00,83.10,92.50


La fuente `igac` tiene 40.547 registros, pero solo el 12% se pudo cruzar con el
catastro. Por eso trabajamos con los registros marcados como `apto_modelo`. En la celda
siguiente, además, los códigos del catastro que no son mediciones (sótano, avalúo cero,
totales de edificio) pasan a vacío.

In [4]:
df = df[df.apto_modelo].copy().reset_index(drop=True)

# códigos del catastro que no son mediciones: se pasan a vacío sin borrar la fila
df.loc[df.piso >= 90, "piso"] = np.nan                     # código de sótano
df.loc[~df.anio_construccion.between(1900, 2025), "anio_construccion"] = np.nan
df.loc[df.avaluo <= 0, "avaluo"] = np.nan                   # sin avalúo
df.loc[df.habitaciones > 20, "habitaciones"] = np.nan       # totales de edificio
df.loc[df.banios > 20, "banios"] = np.nan

df["edad"] = df.anio - df.anio_construccion
df.loc[df.edad < 0, "edad"] = np.nan
df["razon_precio_avaluo"] = df.precio / df.avaluo
df.loc[df.localidad == "SIN LOCALIDAD", "localidad"] = pd.NA
df["estrato"] = pd.Categorical(df.estrato, categories=ORDEN_ESTRATO, ordered=True)

print(f"Base completa: {n_total:,}   apto_modelo: {len(df):,} ({len(df)/n_total:.0%})")

Base completa: 68,709   apto_modelo: 21,953 (32%)


In [5]:
# Herramientas para las pruebas estadísticas. Cada prueba que hacemos queda
# guardada en PRUEBAS y al final se muestran todas juntas (sección 8).
PRUEBAS = []

def p_texto(p):
    return "< 0,001" if p < 0.001 else f"{p:.3f}".replace(".", ",")

def anotar(pregunta, prueba, estadistico, p, efecto=""):
    PRUEBAS.append({"Pregunta": pregunta, "Prueba": prueba,
                    "Estadístico": round(float(estadistico), 3), "p-valor": p_texto(p),
                    "Tamaño del efecto": efecto})
    print(f"{prueba}: estadístico = {estadistico:,.3f}   p = {p_texto(p)}   {efecto}")

def holm(pvalores):
    """Corrección de Holm para comparaciones múltiples."""
    p = np.asarray(pvalores, dtype=float)
    orden = np.argsort(p)
    ajustado = np.empty_like(p)
    acumulado = 0
    for rango, i in enumerate(orden):
        acumulado = max(acumulado, (len(p) - rango) * p[i])
        ajustado[i] = min(acumulado, 1)
    return ajustado

def kruskal_por_grupo(datos, variable, grupo, grupos):
    """Kruskal-Wallis y su épsilon² (proporción de la variación en rangos que explica el grupo)."""
    muestras = [datos.loc[datos[grupo] == g, variable].dropna() for g in grupos]
    muestras = [m for m in muestras if len(m) > 0]
    H, p = stats.kruskal(*muestras)
    n = sum(len(m) for m in muestras)
    return H, p, H / (n - 1)

def v_de_cramer(tabla):
    chi2, p, gl, _ = stats.chi2_contingency(tabla)
    n = tabla.to_numpy().sum()
    return chi2, p, gl, np.sqrt(chi2 / (n * (min(tabla.shape) - 1)))

## Datos faltantes

In [6]:
faltan = df.isna().mean().mul(100)
faltan = faltan[faltan > 0].sort_values()
fig = go.Figure(go.Bar(x=faltan.values, y=faltan.index, orientation="h", marker_color=GRIS,
                       text=[f"{v:.1f}%" for v in faltan.values], textposition="outside",
                       hovertemplate="%{y}: %{x:.1f}% sin dato<extra></extra>"))
fig.update_xaxes(title="% de registros sin dato", range=[0, 80])
mostrar(fig, "Datos faltantes en el universo de trabajo", alto=480)

Lo que más falta son columnas técnicas del cruce con el catastro (`origen_atributos`,
`uso`), que no usamos. De las variables del análisis falta sobre todo la localidad (36%,
contando los registros marcados SIN LOCALIDAD), y después el avalúo y el barrio (9% cada uno).

In [7]:
for col, nombre in [("avaluo", "avalúo"), ("anio_construccion", "año de construcción")]:
    falta = df[col].isna()
    sin, con = df.loc[falta, "precio"], df.loc[~falta, "precio"]
    U, p = stats.mannwhitneyu(sin, con)
    print(f"{nombre}: precio mediano sin dato {cop(sin.median())}, con dato {cop(con.median())}")
    anotar(f"¿El precio cambia cuando falta el {nombre}?", "Mann-Whitney", U, p,
           f"P(sin dato > con dato) = {U / (len(sin) * len(con)):.2f}")

avalúo: precio mediano sin dato $195.0 M, con dato $150.0 M
Mann-Whitney: estadístico = 22,780,982.500   p = < 0,001   P(sin dato > con dato) = 0.58
año de construcción: precio mediano sin dato $205.0 M, con dato $152.2 M
Mann-Whitney: estadístico = 5,221,248.500   p = < 0,001   P(sin dato > con dato) = 0.59


Las ventas sin avalúo o sin año de construcción son más caras que el resto, así que los
faltantes no son al azar y rellenarlos con la mediana sesgaría los resultados. No
imputamos nada en el exploratorio: cada análisis usa los registros que tienen el dato, y
la imputación de la edad se hará dentro del modelo.

## Preparación de la base

### Ajuste por inflación

Una venta de 2017 y una de 2025 no se pueden comparar en pesos corrientes: entre esos
años la inflación acumulada fue de más del 50%. Llevamos todos los precios a pesos de
diciembre de 2025 con el IPC del DANE.

In [8]:
# Variación anual del IPC (diciembre a diciembre, %), DANE
VARIACION_IPC = {2016: 5.75, 2017: 4.09, 2018: 3.18, 2019: 3.80, 2020: 1.61,
                 2021: 5.62, 2022: 13.12, 2023: 9.28, 2024: 5.20, 2025: 5.10}

# índice de diciembre con base diciembre 2018 = 100
ipc_dic = {2018: 100.0}
for a in range(2019, 2026):
    ipc_dic[a] = ipc_dic[a - 1] * (1 + VARIACION_IPC[a] / 100)
for a in range(2017, 2014, -1):
    ipc_dic[a] = ipc_dic[a + 1] / (1 + VARIACION_IPC[a + 1] / 100)

# nivel medio de cada año: media geométrica entre el diciembre anterior y el del año
ipc_anio = pd.Series({a: np.sqrt(ipc_dic[a - 1] * ipc_dic[a]) for a in range(2016, 2026)})
factor = ipc_dic[2025] / ipc_anio

df["precio_real"] = df.precio * df.anio.map(factor)
df["precio_m2_real"] = df.precio_m2 * df.anio.map(factor)
df["log_precio"] = np.log10(df.precio_real)

pd.DataFrame({"inflación del año (%)": pd.Series(VARIACION_IPC),
              "IPC medio (dic 2018 = 100)": ipc_anio.round(1),
              "multiplicar por": factor.round(3)})

,inflación del año (%),IPC medio (dic 2018 = 100),multiplicar por
2016,5.75,90.50,1.68
2017,4.09,95.00,1.60
2018,3.18,98.40,1.55
2019,3.80,101.90,1.49
2020,1.61,104.60,1.46
2021,5.62,108.40,1.41
2022,13.12,118.50,1.28
2023,9.28,131.70,1.16
2024,5.20,141.20,1.08
2025,5.10,148.50,1.02


Un peso de 2019 equivale a 1,49 pesos de 2025. Desde aquí, `precio_real` y
`precio_m2_real` son las variables de precio. Usamos la variación anual oficial y
estimamos el nivel medio de cada año; si se quiere más precisión, se pueden reemplazar
por los índices mensuales del DANE según el mes de la venta.

### Precio por m² fuera de rango

La base trae la bandera `precio_m2_en_rango`, que marca las ventas con un precio por m²
fuera de lo creíble para Barranquilla. No la definimos nosotros: viene de la limpieza
anterior. Aquí vemos qué hay detrás antes de decidir.

In [9]:
viv0 = df[df.clase_unidad_principal == "vivienda"]
fuera = viv0[~viv0.precio_m2_en_rango]
bien = viv0[viv0.precio_m2_en_rango]

# Rango que acepta la bandera de la base
print(f"Rango aceptado: de {cop(bien.precio_m2.min())} a {cop(bien.precio_m2.max())} por m²")
print(f"Viviendas fuera de rango: {len(fuera):,} de {len(viv0):,} ({len(fuera)/len(viv0):.1%})")

baratas = fuera[fuera.precio_m2 < bien.precio_m2.min()]
caras = fuera[fuera.precio_m2 > bien.precio_m2.max()]
pd.DataFrame({
    "ventas": [len(baratas), len(caras)],
    "precio mediano": [cop(baratas.precio.median()), cop(caras.precio.median())],
    "área mediana (m²)": [baratas.area.median(), caras.area.median()],
    "área > 150 m²": [(baratas.area > 150).sum(), (caras.area > 150).sum()],
    "área = suma de pisos": [int(baratas.area_es_suma_pisos.sum()), int(caras.area_es_suma_pisos.sum())],
    "precio < 20 M": [(baratas.precio < 20e6).sum(), (caras.precio < 20e6).sum()],
}, index=["muy baratas por m²", "muy caras por m²"])

Rango aceptado: de $300 mil a $25.0 M por m²
Viviendas fuera de rango: 810 de 19,681 (4.1%)


,ventas,precio mediano,área mediana (m²),área > 150 m²,área = suma de pisos,precio < 20 M
muy baratas por m²,778,$35.0 M,185.57,439,189,214
muy caras por m²,32,$2.30 mil M,59.00,3,0,0


El precio por m² es precio dividido por área, así que el error puede estar en cualquiera
de los dos. Casi todas (778) son muy baratas por m², y la mayoría no por el precio sino
por el área: más de la mitad tienen más de 150 m² y cerca de una cuarta parte tienen el
área sumada de varios pisos. El resto son ventas a precios simbólicos (10 M por un
apartamento), que seguramente son cesiones o errores de digitación. Las 32 muy caras
parecen ventas de edificios enteros registradas como una sola unidad.

In [10]:
def resumen(d):
    return pd.Series({
        "ventas": f"{len(d):,}",
        "precio medio": cop(d.precio_real.mean()),
        "precio mediano": cop(d.precio_real.median()),
        "máximo": cop(d.precio_real.max()),
        "asimetría": round(d.precio_real.skew(), 1),
        "Pearson precio-área": round(d.precio_real.corr(d.area), 2),
    })

pd.DataFrame({"con las fuera de rango": resumen(viv0), "sin ellas": resumen(bien)})

,con las fuera de rango,sin ellas
ventas,"19,681","18,871"
precio medio,$278.9 M,$272.8 M
precio mediano,$175.1 M,$177.1 M
máximo,$94.76 mil M,$23.12 mil M
asimetría,81.30,20.90
Pearson precio-área,0.07,0.59


Son el 4% de las viviendas, pero cambian mucho las medidas sensibles a extremos: la
asimetría baja de 81 a 21 y la correlación entre precio y área pasa de 0,07 a 0,59. La
mediana casi no cambia. Las quitamos porque, si el área está mal, el registro no sirve
ni para describir ni para el modelo. El corte inferior (300 mil por m²) viene de la
limpieza anterior; conviene justificarlo en el documento o revisarlo para casas viejas
de estratos 1 y 2.

### Registros duplicados

In [11]:
COLS = [c for c in df.columns if c != "id"]
copias = bien.duplicated(subset=COLS)
viv = bien[~copias].copy()

alameda = bien.barrio == "SECTOR ALAMEDA DEL RIO"
print(f"Copias quitadas: {copias.sum():,}  ({alameda[copias].mean():.0%} de Alameda del Río)")
print(f"Peso de Alameda del Río: {alameda.mean():.1%} con copias, {alameda[~copias].mean():.1%} sin copias")

# columnas de apoyo para los gráficos
viv["estrato_txt"] = viv.estrato.astype(str).map(dict(zip(ESTRATOS, NOMBRES_ESTRATO))).fillna("Sin estrato")
viv["barrio_txt"] = viv.barrio.fillna("sin barrio").str.title()
viv["precio_txt"] = viv.precio_real.map(cop)
viv["precio_M"] = viv.precio_real / 1e6
viv["pm2_M"] = viv.precio_m2_real / 1e6

pasos = pd.Series({"Base completa": n_total, "apto_modelo": len(df), "Viviendas": len(viv0),
                   "Precio por m² en rango": len(bien), "Sin copias": len(viv)})
fig = go.Figure(go.Funnel(y=pasos.index, x=pasos.values, textinfo="value+percent initial",
                          marker_color=[GRIS, GRIS, AZUL, AZUL, ROJO]))
mostrar(fig, "De la base completa a la base de análisis", alto=380)

Copias quitadas: 3,412  (88% de Alameda del Río)
Peso de Alameda del Río: 29.7% con copias, 16.4% sin copias


Se quitan 3.412 filas repetidas, casi todas de Alameda del Río, cuyo peso baja del 30%
al 16%. Quedan 15.459 viviendas, y todos los capítulos siguientes usan esta misma base.

## Ventas por año y fuente

In [12]:
t = viv.groupby(["anio", "fuente"]).size().rename("ventas").reset_index()
fig = px.bar(t, x="anio", y="ventas", color="fuente", barmode="stack",
             labels={"anio": "año", "fuente": "fuente"})
fig.update_xaxes(dtick=1)
mostrar(fig, "Ventas de vivienda por año y fuente")

Antes de 2019 hay muy pocas ventas y 2024 es un año flojo. La mayoría de los datos son
del observatorio.